In [ ]:
library(Seurat)
library(pheatmap)
library(RColorBrewer)
library(ComplexHeatmap)
library(tidyr)
library(ggplot2)
library(dplyr)
library(cowplot)
library(data.table)
library(harmony)
library(ggpubr)
library(ggpmisc)
set.seed(123)
library(tidyverse)
library(ggsci)
library(patchwork)
library(rhdf5)
library(future)
options(future.globals.maxSize = 50 * 1024^3)
plan(multisession, workers=16)
library(viridis)

In [ ]:
output = './02.visualization_result/'

In [ ]:
cols_celltype = c("#DC143C","#0000FF","#20B2AA","#FFA500","#9370DB","#98FB98","#E8E8E8")
names(cols_celltype) = c('EPI','PE/HYPO','TE','INTER','NR','ICM','Other')
cols_species = c("#DC143C","#0000FF","#20B2AA","#FFA500","#9370DB","#98FB98","#F08080","#1E90FF","#E8E8E8")
names(cols_species) = c('human_blastoid','human_blastocyst_E5','human_blastocyst_E6','human_blastocyst_E7',
                        'chimpanzee_blastoid','macaque_blastocyst','Orangutan_blastoid','Orangutan_bi_blastoid',
                        'Other')

In [6]:

###################read data######################


In [ ]:
Data_all = readRDS(paste0('./01.integration_result/',"/","CrossSpecies_CCA.rds"))

In [ ]:
options(repr.plot.width=10, repr.plot.height=10)
p = DimPlot(Data_all, reduction = "umap.cca",label = FALSE,group.by= "New_ann",cols=c(cols_celltype),pt.size =0.5)+coord_fixed()+ 
    theme_minimal()+theme(panel.grid.major = element_blank(), 
        panel.grid.minor = element_blank(), 
        panel.border = element_blank(), 
        axis.title = element_blank(),  
        axis.text = element_blank())+
  theme()+labs(title = "")
print(p)
png(paste0(output,'/','AllData_integrated_UMAP_celltype.png'),width = 1800, height = 1800,res = 300)
print(p)
dev.off()

In [ ]:

################split to different species##################


In [ ]:
data_list = c('human_blastoid','human_blastocyst_E5',
              'human_blastocyst_E6','human_blastocyst_E7',
              'chimpanzee_blastoid','macaque_blastocyst',
             'Orangutan_blastoid','Orangutan_bi_blastoid')
for(i in data_list){
    seuobj = Data_all
    seuobj$test = seuobj$New_ann
    seuobj$test[which(seuobj$label != i)] = 'Other'
    seuobj$test = factor(seuobj$test,levels = c('Other','EPI','PE/HYPO','TE','INTER','NR','ICM'))
    options(repr.plot.width=7, repr.plot.height=7)
    p = DimPlot(seuobj, reduction = "umap.cca",label = FALSE,group.by= "test",cols=c(cols_celltype),pt.size =0.5,order = TRUE)+coord_fixed()+ 
    theme_minimal()+theme(panel.grid.major = element_blank(),
        panel.grid.minor = element_blank(), 
        panel.border = element_blank(), 
        axis.title = element_blank(),
        axis.text = element_blank())+
      theme()+labs(title = i)
    print(p)
    png(paste0(output,'/',i,'_integrated_UMAP_celltype.png'),width = 1800, height = 1800,res = 300)
    print(p)
    dev.off()
    
}

In [ ]:

#########################cell propotion bar plot#######################


In [ ]:
Date_subset = subset(Data_all, subset = label %in% c('human_blastocyst_E5','human_blastocyst_E6','human_blastocyst_E7',
                                                 'human_blastoid','Orangutan_blastoid','Orangutan_bi_blastoid'))
Data = Date_subset
#calculate propotion
meta = Data@meta.data
plot_data = meta %>%
  group_by(label, New) %>%
  summarise(count = n()) %>%
  ungroup() %>%
  group_by(label) %>%
  mutate(prop = count / sum(count)) 
plot_data$label = factor(plot_data$label, levels = c('human_blastocyst_E5','human_blastocyst_E6','human_blastocyst_E7',
                                                     'human_blastoid','Orangutan_blastoid','Orangutan_bi_blastoid'
                                                     ))
plot_data$New_annotation_Nicole = factor(plot_data$New, levels = c('NR','INTER','PE/HYPO',
                                                     'TE','EPI','ICM'))
options(repr.plot.width=6, repr.plot.height=8)
p = ggplot(plot_data, aes(x = label, y = prop, fill = New_annotation_Nicole)) +
  geom_bar(stat = "identity") +
  scale_fill_manual(values = cols_celltype) +
  scale_y_continuous(labels = scales::percent_format()) +
  labs(x = "", y = "", fill = "Cell Type") +
  theme_classic() +
  theme(
    axis.text.x = element_text(angle = 90, hjust = 1, size = 14),  
    axis.text.y = element_text(size = 14),                         
    axis.title.x = element_text(size = 16, face = "bold"),          
    axis.title.y = element_text(size = 16, face = "bold"),          
    legend.title = element_text(size = 14, face = "bold"),          
    legend.text = element_text(size = 12)                         
  )
print(p)
pdf(file = paste0(output,'/',"Barplot_cell_propotion_human2Orangutan.pdf"), width = 6, height = 6)
print(p)
dev.off()